In [1]:
import pandas as pd


In [6]:
df_mvp = pd.read_excel('data_files/mvp_2003_2024_votes.xlsx', sheet_name='2003-2004')  
df_mvp.head()

,Rank,Player,Age,Tm,First,Pts Won,Pts Max,Share,G,MP,PTS,TRB,AST,STL,BLK,FG%,3P%,FT%,WS,WS/48
0,1.0,Kevin Garnett,27.0,MIN,120.0,1219.0,1230.0,0.991,82.0,39.4,24.2,13.9,5.0,1.5,2.2,0.499,0.256,0.791,18.3,0.272
1,2.0,Tim Duncan,27.0,SAS,0.0,716.0,1230.0,0.582,69.0,36.6,22.3,12.4,3.1,0.9,2.7,0.501,0.167,0.599,13.1,0.249
2,3.0,Jermaine O'Neal,25.0,IND,2.0,523.0,1230.0,0.425,78.0,35.7,20.1,10.0,2.1,0.8,2.6,0.434,0.111,0.757,9.0,0.155
3,4.0,Peja StojakoviÄ‡,26.0,SAC,1.0,281.0,1230.0,0.228,81.0,40.3,24.2,6.3,2.1,1.3,0.2,0.480,0.433,0.927,13.5,0.198
4,5.0,Kobe Bryant,25.0,LAL,0.0,212.0,1230.0,0.172,65.0,37.6,24.0,5.5,5.1,1.7,0.4,0.438,0.327,0.852,10.7,0.210


In [16]:
df_teamrec = pd.read_excel('data_files/team records.xlsx', sheet_name='2003-2004')  
df_teamrec.head()

,Eastern Conference,W,L,W/L%,GB,PS/G,PA/G,SRS
0,Atlantic Division,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,New Jersey Nets*,47.0,35.0,0.573,—,90.3,87.8,1.88
2,Miami Heat*,42.0,40.0,0.512,5.0,90.3,89.7,-0.13
3,New York Knicks*,39.0,43.0,0.476,8.0,92.0,93.5,-1.97
4,Boston Celtics*,36.0,46.0,0.439,11.0,95.3,96.7,-1.99


In [73]:
# Create new Excel where the Ws are sorted for each year, each conference
all_sheets = pd.read_excel('data_files/team records.xlsx', sheet_name=None, header=None)

team_abbreviations = {
    'Atlanta Hawks': 'ATL', 'Boston Celtics': 'BOS', 'Brooklyn Nets': 'BKN', 'Charlotte Hornets': 'CHA',
    'Chicago Bulls': 'CHI', 'Cleveland Cavaliers': 'CLE', 'Dallas Mavericks': 'DAL', 'Denver Nuggets': 'DEN',
    'Detroit Pistons': 'DET', 'Golden State Warriors': 'GSW', 'Houston Rockets': 'HOU', 'Indiana Pacers': 'IND',
    'Los Angeles Clippers': 'LAC', 'Los Angeles Lakers': 'LAL', 'Memphis Grizzlies': 'MEM', 'Miami Heat': 'MIA',
    'Milwaukee Bucks': 'MIL', 'Minnesota Timberwolves': 'MIN', 'New Orleans Pelicans': 'NOP', 'New York Knicks': 'NYK',
    'Oklahoma City Thunder': 'OKC', 'Orlando Magic': 'ORL', 'Philadelphia 76ers': 'PHI', 'Phoenix Suns': 'PHX',
    'Portland Trail Blazers': 'POR', 'Sacramento Kings': 'SAC', 'San Antonio Spurs': 'SAS', 'Toronto Raptors': 'TOR',
    'Utah Jazz': 'UTA', 'Washington Wizards': 'WAS', 'Charlotte Bobcats':'CHA', 'New Orleans Hornets': 'NOH','New Orleans/Oklahoma City Hornets':'NOH',  'New Jersey Nets': 'NJN', 'Seattle SuperSonics': 'SEA'
}
cleaned_sheets = {}
for sheet_name, df in all_sheets.items():
    # Set proper headers
    df.columns = df.iloc[0]
    df = df.iloc[1:].reset_index(drop=True)

    # Find the index of the row where "Western Conference" appears
    western_index = df[df['Eastern Conference'] == 'Western Conference'].index[0]

    # Split data into Eastern and Western Conference
    eastern_df = df.iloc[0:western_index].sort_values(by='W', ascending=False)
    western_df = df.iloc[western_index+1:].sort_values(by='W', ascending=False)

    # Add ranking columns
    eastern_df['Conference Ranking'] = range(1, len(eastern_df) + 1)
    western_df['Conference Ranking'] = range(1, len(western_df) + 1)
    eastern_df = eastern_df.dropna(subset=["W"])
    western_df = western_df.dropna(subset=["W"])
    # Add 'Tm' column with abbreviations
    eastern_df['Tm'] = eastern_df['Eastern Conference'].str.replace(r'\*', '', regex=True).str.strip().map(team_abbreviations)
    western_df['Tm'] = western_df['Eastern Conference'].str.replace(r'\*', '', regex=True).str.strip().map(team_abbreviations)
    # Combine the sorted sections back into the full sheet
    cleaned_sheets[sheet_name] = pd.concat([eastern_df, western_df]).reset_index(drop=True)

# Save to new Excel file
output_path = 'data_files/team_records_cleaned.xlsx'
with pd.ExcelWriter(output_path) as writer:
    for sheet_name, df in cleaned_sheets.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"Ranked data saved to {output_path}")


Ranked data saved to data_files/team_records_cleaned.xlsx
